# Human-in-the-loop
Pause before a high-stakes tool call, let a human approve, edit, or reject it, then resume exactly where the agent left off.

## Concept
HumanInTheLoopMiddleware names which tools need sign-off via interrupt_on. When one fires, the run pauses; you resume with Command(resume=...). A checkpointer is required so the paused state has somewhere to wait.

In [4]:
from pydantic import SecretStr
import os

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.tools import tool

@tool
def search(query: str) -> str:
    """Search the web or knowledge base for information."""
    # Dummy implementation
    return f"Mock search results for: '{query}' -> Found 3 relevant items."

@tool
def send_email(to: str, subject: str, body: str = "") -> str:
    """Send an email to a recipient with a subject and body."""
    # Dummy implementation
    print(f"[SIMULATED EMAIL SENT]\nTo: {to}\nSubject: {subject}\nBody: {body}\n")
    return f"Email successfully sent to {to} with subject '{subject}'."

@tool
def delete_database(db_name: str = "default_db") -> str:
    """Delete a database (high-risk destructive action)."""
    # Dummy implementation
    print(f"[SIMULATED ACTION] Database '{db_name}' has been wiped.")
    return f"Database '{db_name}' successfully deleted."

model = init_chat_model(
    model=os.environ["FLEXAI_MODEL"],
    base_url=os.environ["FLEXAI_URL"],
    api_key=SecretStr(os.environ["FLEXAI_API_KEY"])
)

## Gate the sensitive tools

In [5]:
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=model,
    tools=[search, send_email, delete_database],
    middleware=[HumanInTheLoopMiddleware(
        interrupt_on={
            "send_email": True,          # needs approval
            "delete_database": True,
            "search": False,             # auto-approved
        }
    )],
    checkpointer=InMemorySaver()         # required to persist across the pause
)

## Run, then resume with a decision

In [27]:
from langgraph.types import Command

config = {"configurable": {"thread_id": "session-2"}}

# pauses at the first gated tool
agent.invoke({"messages": [{"role": "user", "content": "Email the team. recipients: test@email.com, subject: test subject, body: hello there"}]}, config)

agent.invoke(Command(
    resume={
        "decisions": [{"type": "approve"}]
    }
), config)

[SIMULATED EMAIL SENT]
To: test@email.com
Subject: test subject
Body: hello there



{'messages': [HumanMessage(content='Email the team. recipients: test@email.com, subject: test subject, body: hello there', additional_kwargs={}, response_metadata={}, id='e7304731-1846-4a0b-887a-e38591d201cb'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 80, 'prompt_tokens': 86, 'total_tokens': 166, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 61, 'rejected_prediction_tokens': None, 'text_tokens': 19}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 80, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-oss-120b', 'system_fingerprint': None, 'id': 'chatcmpl-ea90ec46-df4b-422b-a52c-01b40105c95e', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1646f-f43b-7782-a9c4-3e09906da64b-0', tool_calls=[{'name': 'send_email', 'args': {'to': 'test@email.com', 'sub

## Inspect what's pending before deciding

In [28]:
config = {"configurable": {"thread_id": "session-1"}}

# pauses at the first gated tool
result = agent.invoke({"messages": [{"role": "user", "content": "Email the team. recipients: test@email.com, subject: test subject, body: hello there"}]}, config)

# the request(s) awaiting a human are on __interrupt__
for req in result["__interrupt__"]:
    print(req.value)     # tool name + args the agent wants to run

# or check the paused snapshot directly
# print(agent.get_state(config).next)   # the node it's waiting on

KeyError: '__interrupt__'

## Approve

In [15]:
agent.invoke(Command(
    resume={
        "decisions":[
            {"type": "approve"}
        ]
    }
), config)

[SIMULATED EMAIL SENT]
To: test@email.com
Subject: test subject
Body: hello there



{'messages': [HumanMessage(content='Email the team', additional_kwargs={}, response_metadata={}, id='a43c4720-69bb-450f-951a-d9ef498b5274'),
  AIMessage(content='Sure thing! Could you let me know the following details so I can send the email correctly?\n\n1. **Recipient(s)** – the email address(es) of the team members (or a distribution list).  \n2. **Subject** – what you’d like the email’s subject line to be.  \n3. **Message body** – the content you want included in the email (feel free to paste it here).  \n\nOnce I have that information, I’ll send the email right away.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 164, 'prompt_tokens': 70, 'total_tokens': 234, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 55, 'rejected_prediction_tokens': None, 'text_tokens': 109}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 80, 'image_tokens

## Edit the tool arguments
Let the reviewer fix the call — the agent runs the tool with the edited args instead.

In [29]:
# Check how many interruptions are pending
for req in result.get("__interrupt__", []):
    print("Pending tool:", req.value)
agent.invoke(
    Command(resume={"decisions": [
        {
            "type": "edit",
            "args": {"to": "team@acme.com", "subject": "Q3 results (reviewed)"},
        }
    ]}),
    config,
)

{'messages': [HumanMessage(content='Email the team', additional_kwargs={}, response_metadata={}, id='a43c4720-69bb-450f-951a-d9ef498b5274'),
  AIMessage(content='Sure thing! Could you let me know the following details so I can send the email correctly?\n\n1. **Recipient(s)** – the email address(es) of the team members (or a distribution list).  \n2. **Subject** – what you’d like the email’s subject line to be.  \n3. **Message body** – the content you want included in the email (feel free to paste it here).  \n\nOnce I have that information, I’ll send the email right away.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 164, 'prompt_tokens': 70, 'total_tokens': 234, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 55, 'rejected_prediction_tokens': None, 'text_tokens': 109}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 80, 'image_tokens

## Reject and keep going
Decline the action; the agent gets the rejection and continues without running the tool.

In [30]:
agent.invoke(
    Command(resume={"decisions": [{
        "type": "reject",
        "message": "Don't send external email — summarize for me instead.",
    }]}),
    config,
)

{'messages': [HumanMessage(content='Email the team', additional_kwargs={}, response_metadata={}, id='a43c4720-69bb-450f-951a-d9ef498b5274'),
  AIMessage(content='Sure thing! Could you let me know the following details so I can send the email correctly?\n\n1. **Recipient(s)** – the email address(es) of the team members (or a distribution list).  \n2. **Subject** – what you’d like the email’s subject line to be.  \n3. **Message body** – the content you want included in the email (feel free to paste it here).  \n\nOnce I have that information, I’ll send the email right away.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 164, 'prompt_tokens': 70, 'total_tokens': 234, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 55, 'rejected_prediction_tokens': None, 'text_tokens': 109}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 80, 'image_tokens

# Watch for the pause while streaming
In a live demo, stream the run and react the moment it interrupts.

In [38]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "Delete the default database"}]},
    config,
    stream_mode="updates",
):
    if "__interrupt__" in chunk:
        print("⏸ waiting for approval:", chunk["__interrupt__"])
        break

⏸ waiting for approval: (Interrupt(value={'action_requests': [{'name': 'delete_database', 'args': {}, 'description': 'Tool execution requires approval\n\nTool: delete_database\nArgs: {}'}], 'review_configs': [{'action_name': 'delete_database', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='ec03f4ba0b7126e6300cb5b013236f92', response_schema=None),)
